In [5]:
%idle_timeout 30
%glue_version 4.0
%worker_type G.1X
%number_of_workers 2

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Current idle_timeout is None minutes.
idle_timeout has been set to 30 minutes.
Setting Glue version to: 4.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 2


In [1]:
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.dynamicframe import DynamicFrame
from awsglue.job import Job

print('Bibliotecas importadas')

Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 2
Idle Timeout: 30
Session ID: 10f1b49d-3f57-4b8e-958f-988058e5b8a2
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session 10f1b49d-3f57-4b8e-958f-988058e5b8a2 to get into ready status...
Session 10f1b49d-3f57-4b8e-958f-988058e5b8a2 has been created.
Bibliotecas importadas


In [2]:
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

job = Job(glueContext)
job.init("teste-interativo-gold")

DATABASE = "state_of_data_db"
BUCKET = "pos-tech-state-of-data-grupo-72"

print("Consultando dados da camada Silver")

df_silver = glueContext.create_dynamic_frame.from_catalog(
    database=DATABASE,
    table_name="silver_state_of_data"
).toDF()

df_silver.createOrReplaceTempView("silver_view")

print('Silver lida e Temp view criada')

Consultando dados da camada Silver
Silver lida e Temp view criada
/opt/amazon/spark/python/lib/pyspark.zip/pyspark/sql/dataframe.py:127: UserWarning: DataFrame constructor is internal. Do not directly use it.


In [3]:
print('Criando camada Gold com Spark.sql')

df_gold = spark.sql("""
    SELECT
        ano,
        id_resposta,
        MAX(CASE WHEN codigo_1 = '1' AND codigo_2 = 'a' AND codigo_3 IS NULL THEN resposta END) AS idade,
        MAX(CASE WHEN codigo_1 = '1' AND codigo_2 = 'b' AND codigo_3 IS NULL THEN resposta END) AS genero,
        MAX(CASE WHEN codigo_1 = '1' AND codigo_2 = 'c' AND codigo_3 IS NULL THEN resposta END) AS cor_raca,
        MAX(CASE WHEN codigo_1 = '1' AND codigo_2 = 'd' AND codigo_3 IS NULL THEN resposta END) AS pcd,
        MAX(CASE WHEN codigo_1 = '1' AND codigo_2 = 'i' AND codigo_3 IS NULL THEN resposta END) AS estado,
        MAX(CASE WHEN codigo_1 = '1' AND codigo_2 = 'm' AND codigo_3 IS NULL THEN resposta END) AS area_formacao,
        MAX(CASE WHEN codigo_1 = '1' AND codigo_2 = 'l' AND codigo_3 IS NULL THEN resposta END) AS nivel_ensino,
        MAX(CASE WHEN codigo_1 = '2' AND codigo_2 = 'b' AND codigo_3 IS NULL THEN resposta END) AS setor,
        MAX(CASE WHEN codigo_1 = '2' AND codigo_2 = 'e' AND codigo_3 IS NULL THEN resposta END) AS e_gestor,
        MAX(CASE WHEN codigo_1 = '2' AND codigo_2 = 'f' AND codigo_3 IS NULL THEN resposta END) AS cargo,
        MAX(CASE WHEN codigo_1 = '2' AND codigo_2 = 'g' AND codigo_3 IS NULL THEN resposta END) AS nivel,
        MAX(CASE WHEN codigo_1 = '2' AND codigo_2 = 'h' AND codigo_3 IS NULL THEN resposta END) AS faixa_salarial,
        MAX(CASE WHEN codigo_1 = '2' AND codigo_2 = 'i' AND codigo_3 IS NULL THEN resposta END) AS tempo_exp
    FROM silver_view
    GROUP BY ano, id_resposta
""")

# Salvando no S3 e catalogando no Data Catalog

caminho_gold = f"s3://{BUCKET}/gold/dimensao_respondente/"

# Limpa qualquer dado anterior antes de escrever (evita duplicação por múltiplas execuções)
glueContext.purge_s3_path(caminho_gold, options={"retentionPeriod": 0})

dyf_gold = DynamicFrame.fromDF(df_gold, glueContext, "dyf_gold")

sink = glueContext.getSink(
    connection_type="s3",
    path=caminho_gold,
    enableUpdateCatalog=True
)
sink.setFormat("glueparquet")
sink.setCatalogInfo(catalogDatabase=DATABASE, catalogTableName="gold_dimensao_respondente")
sink.writeFrame(dyf_gold)

print("Gold gravada e catalogada com sucesso.")
job.commit()

Criando camada Gold com Spark.sql
Gold gravada e catalogada com sucesso.
